# 01 – Bronze ➜ Silver (ProductIQ, Contoso Home Appliances – synthetic data)

This notebook cleans the raw **bronze** CSV extracts and writes conformed **silver** Delta tables (`silver_*`).

* **Input:** `Files/bronze/<source_table>/<source_table>.csv` in the default lakehouse **productiq_lakehouse**
  (uploaded by the deployment scripts from `data/output/bronze`).
* **Output:** Delta tables `silver_regions`, `silver_product_aliases`, `silver_products`, `silver_plants`, `silver_suppliers`,
  `silver_components`, `silver_bom`, `silver_customers`, `silver_sales`, `silver_reviews`, `silver_tickets`, `silver_warranty`,
  `silver_supplier_capacity`, `silver_component_costs`, `silver_shipments`, `silver_market_signals`.

**Cleaning rules** (documented in `docs/DATA-MODEL.md`):
1. De-duplicate on the business key (re-sent extracts).
2. Normalise region codes (trim, upper case, ISO-3 / legacy aliases such as `JPN`, `USA`, `GB` ➜ `JP`, `US`, `UK`).
3. Map legacy product codes (e.g. `RC-R7`) to the canonical `product_code` via `erp_product_aliases`; drop unknown products.
4. Explicit type casting (dates, timestamps, numbers, booleans with mixed spellings `Y/yes/1/TRUE`).
5. Null handling: recompute missing `net_revenue_usd`, `utilisation_pct`, `co2e_kg`, `purchase_date`; default categories.
6. Text enrichment for reviews and tickets: `language`, `pet_owner` (profile flag ➜ pet mentions in text ➜ CRM segment ➜ false),
   `topic` (ordered keyword rules, English + Japanese) and `sentiment_score` / `sentiment_label` (rating + lexicon).
7. Market signals: keep the latest published revision per month × region × category × metric.

The notebook is deterministic and idempotent (all writes use `overwrite`). The same logic is mirrored in pandas in
`data/generator/reference_pipeline.py`, which the data generator uses to validate the scenario signals.

In [ ]:
# Configuration – attach the lakehouse "productiq_lakehouse" as the DEFAULT lakehouse before running.
from pyspark.sql import functions as F
from pyspark.sql import Window

BRONZE_ROOT = "Files/bronze"


def read_bronze(name: str):
    """Read one bronze CSV with all columns as strings; explicit casts below keep the run deterministic."""
    return (spark.read.format("csv")
            .option("header", True)
            .option("inferSchema", False)
            .option("multiLine", True)      # review / ticket texts may contain line breaks
            .option("quote", '"')
            .option("escape", '"')          # RFC 4180 doubled quotes
            .option("encoding", "UTF-8")    # Japanese reviews
            .load(f"{BRONZE_ROOT}/{name}/{name}.csv"))


def write_table(df, name: str):
    """Overwrite a Delta table in the default lakehouse and print its row count."""
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(name)
    print(f"{name:<32}{spark.table(name).count():>12,} rows")

In [ ]:
# BEGIN SHARED RULES (identical copy of data/generator/reference_pipeline.py)
REGION_ALIASES = {
    "JPN": "JP", "JAPAN": "JP", "USA": "US", "U.S.": "US", "GB": "UK", "GBR": "UK",
    "DEU": "DE", "GER": "DE", "FRA": "FR", "KOR": "KR", "CHN": "CN", "SGP": "SG",
    "AUS": "AU", "CAN": "CA",
}

TRUE_VALUES = ["true", "t", "yes", "y", "1"]
FALSE_VALUES = ["false", "f", "no", "n", "0"]

# Ordered keyword rules: the first matching topic wins.  Patterns are Java/Python
# compatible regular expressions applied to the lower-cased "title + text".
TOPIC_RULES = [
    ("hair_tangling", [r"tangl", r"(?:wrap|wound|wind)\w* around", r"hair.{0,40}brush",
                       r"brush.{0,40}hair", r"\bfur\b.{0,40}brush", r"brush.{0,40}\bfur\b",
                       "絡ま", "絡み", "ブラシに髪", "ブラシに毛"]),
    ("dock_emptying", [r"self-empt", r"auto-empt", r"emptying", r"dust bag", r"empty station",
                       r"dock.{0,30}(?:clog|jam|full|empt)", "ゴミ収集", "紙パック"]),
    ("navigation", [r"navigat", r"\bbump", r"stuck under", r"\bmaps?\b", r"mapping", r"obstacle",
                    r"lidar", r"gets lost", r"got lost", r"cliff sensor", "マップ", "マッピング",
                    "ぶつか", "障害物", "段差"]),
    ("battery_life", [r"batter", r"runtime", r"run time", r"one charge", r"recharg",
                      r"holding charge", "バッテリー", "電池", "充電"]),
    ("suction", [r"suction", r"pick(?:s|ed|ing)? up", r"leaves debris", "吸引", "吸い込"]),
    ("filter", [r"filter", r"\bhepa\b", "フィルター"]),
    ("noise", [r"noise", r"noisy", r"\bloud", r"quiet", r"decibel", r"rattl", r"whin(?:e|ing)",
               "うるさい", "騒音", "音が大き", "静か"]),
    ("app_connectivity", [r"\bapp\b", r"wi-?fi", r"connect", r"firmware", r"pairing",
                          r"bluetooth", "アプリ", "接続"]),
    ("water_leak", [r"leak", r"puddle", "水漏れ"]),
    ("cooking_quality", [r"\brice\b", r"cook", r"heats unevenly", r"uneven heating", r"reheat",
                         r"defrost", r"undercooked", r"burnt", "炊", "ご飯", "加熱", "温め"]),
    ("cooling", [r"cooling", r"not cold", r"warm inside", r"temperature", r"freez",
                 r"ice maker", r"frost", "冷え", "冷蔵", "冷凍"]),
    ("wash_performance", [r"stain", r"rinse", r"spots on", r"residue", r"still dirty",
                          r"wash cycle", r"wash results", r"spotless", "汚れが落ち", "洗浄",
                          "すすぎ"]),
    ("durability", [r"\bbroke", r"stopped working", r"died after", r"fell apart", r"crack",
                    r"defect", r"durab", r"sturdy", r"build quality", r"well built", "壊れ",
                    "故障", "丈夫", "しっかり"]),
    ("price", [r"price", r"pricey", r"expensive", r"overpriced", r"value for money",
               r"worth the money", r"\bcheap", "価格", "値段", "コスパ"]),
    ("delivery", [r"deliver", r"shipping", r"courier", r"arrived", r"packag", r"parcel",
                  "配送", "届", "梱包"]),
    ("ease_of_use", [r"easy to use", r"easy to set up", r"set ?up", r"intuitive", r"controls",
                     r"manual", r"confusing", r"buttons", "使いやす", "操作", "説明書"]),
]
DEFAULT_TOPIC = "other"

# Sentiment lexicon (presence of each term counts once).
POS_TERMS = [r"\bgreat\b", r"\bexcellent\b", r"\blove[sd]?\b", r"\bperfect(?:ly)?\b",
             r"\bamazing\b", r"\bimpressed\b", r"\brecommend", r"\bfantastic\b",
             r"\bsatisfied\b", r"\bhappy\b", r"\bflawless(?:ly)?\b", r"\breliable\b",
             r"\bawesome\b", r"\bsuperb\b", r"\bgame changer\b",
             "満足", "最高", "素晴らしい", "便利", "気に入", "おすすめ", "買ってよかった", "助か"]
NEG_TERMS = [r"\bdisappoint", r"\bterrible\b", r"\bawful\b", r"\bpoor(?:ly)?\b",
             r"\bbroken?\b", r"\buseless\b", r"\bannoying\b", r"\bfrustrat", r"\bworst\b",
             r"\bwaste\b", r"\brefund", r"\breturn(?:ed|ing)?\b", r"\bhate\b", r"\bavoid\b",
             r"\bproblems?\b", r"\bissues?\b", r"\bweak\b", r"\bfail(?:s|ed|ing)?\b",
             "残念", "がっかり", "最悪", "不満", "返品", "困", "大変", "ひどい"]

# Mentions that identify a pet-owning customer in free text.
PET_TERMS = [r"\bdogs?\b", r"\bcats?\b", r"\bpets?\b", r"\bpupp(?:y|ies)\b", r"\bkittens?\b",
             "ペット", "犬", "猫"]

# Any Hiragana / Katakana / CJK ideograph => Japanese text.
JA_CHAR_PATTERN = r"[\u3040-\u30ff\u4e00-\u9fff]"

# Sentiment = 0.75 * rating component + 0.25 * lexicon score; tickets have no
# rating and use TICKET_RATING_COMPONENT (they are complaints by nature).
RATING_WEIGHT = 0.75
LEXICON_WEIGHT = 0.25
TICKET_RATING_COMPONENT = -0.5
POSITIVE_THRESHOLD = 0.25
NEGATIVE_THRESHOLD = -0.25

# kg CO2e per tonne-km by transport mode (used when the source value is missing).
EMISSION_FACTORS_KG_PER_TKM = {"ocean": 0.016, "rail": 0.028, "truck": 0.105, "air": 0.602}
# END SHARED RULES


def combined_pattern(patterns):
    """Join regexes into one alternation (same string as the pandas reference)."""
    return "|".join(f"(?:{p})" for p in patterns)


TOPIC_PATTERNS = [(t, combined_pattern(p)) for t, p in TOPIC_RULES]
PET_PATTERN = combined_pattern(PET_TERMS)

In [ ]:
# Reusable column expressions


def upper_trim(c):
    return F.upper(F.trim(F.col(c)))


def lower_trim(c):
    return F.lower(F.trim(F.col(c)))


def norm_region(c):
    """Trim + upper case + alias mapping (JPN -> JP, USA -> US, GB -> UK, ...)."""
    up = F.upper(F.trim(F.col(c)))
    expr = up
    for alias, code in REGION_ALIASES.items():
        expr = F.when(up == F.lit(alias), F.lit(code)).otherwise(expr)
    return expr


def to_bool(c):
    """Map mixed boolean spellings (true/Y/yes/1, false/N/no/0) to boolean; anything else -> null."""
    low = F.lower(F.trim(F.col(c)))
    return (F.when(low.isin(TRUE_VALUES), F.lit(True))
             .when(low.isin(FALSE_VALUES), F.lit(False))
             .otherwise(F.lit(None).cast("boolean")))


def topic_expr(text):
    """First matching keyword rule wins; 'other' when nothing matches."""
    expr = F.lit(DEFAULT_TOPIC)
    for topic, pattern in reversed(TOPIC_PATTERNS):
        expr = F.when(text.rlike(pattern), F.lit(topic)).otherwise(expr)
    return expr


def lexicon_expr(text):
    """(positive terms present - negative terms present) / max(total, 1), in -1..1."""
    pos = sum((F.when(text.rlike(p), 1).otherwise(0) for p in POS_TERMS), F.lit(0))
    neg = sum((F.when(text.rlike(p), 1).otherwise(0) for p in NEG_TERMS), F.lit(0))
    return (pos - neg) / F.greatest(pos + neg, F.lit(1))


def add_sentiment(df, rating_component, text):
    raw = F.lit(RATING_WEIGHT) * rating_component + F.lit(LEXICON_WEIGHT) * lexicon_expr(text)
    return (df.withColumn("_raw", raw)
              .withColumn("sentiment_score", F.round(F.col("_raw"), 4))
              .withColumn("sentiment_label",
                          F.when(F.col("_raw") > F.lit(POSITIVE_THRESHOLD), "positive")
                           .when(F.col("_raw") < F.lit(NEGATIVE_THRESHOLD), "negative")
                           .otherwise("neutral"))
              .drop("_raw"))


def add_language(df, text_col):
    detected = F.when(F.col(text_col).rlike(JA_CHAR_PATTERN), F.lit("ja")).otherwise(F.lit("en"))
    return df.withColumn("language", F.coalesce(lower_trim("language"), detected))

## Reference data: regions, product aliases, products, plants, suppliers, components, BOM

In [ ]:
regions = (read_bronze("erp_regions")
           .withColumn("region_code", norm_region("region_code"))
           .withColumn("fx_to_usd", F.col("fx_to_usd").cast("double"))
           .dropDuplicates(["region_code"]))
write_table(regions, "silver_regions")

aliases = (read_bronze("erp_product_aliases")
           .withColumn("alias_code", upper_trim("alias_code"))
           .withColumn("product_code", upper_trim("product_code"))
           .dropDuplicates(["alias_code"]))
write_table(aliases, "silver_product_aliases")

products = (read_bronze("erp_products")
            .withColumn("product_code", upper_trim("product_code"))
            .withColumn("launch_date", F.to_date("launch_date", "yyyy-MM-dd"))
            .withColumn("discontinue_date", F.to_date("discontinue_date", "yyyy-MM-dd"))
            .withColumn("list_price_usd", F.col("list_price_usd").cast("double"))
            .withColumn("weight_kg", F.col("weight_kg").cast("double")))
for c in ["has_lidar", "has_self_empty_dock", "has_mop", "is_pet_edition"]:
    products = products.withColumn(c, F.coalesce(to_bool(c), F.lit(False)))
products = products.dropDuplicates(["product_code"])
write_table(products, "silver_products")

ALIAS_DF = spark.table("silver_product_aliases").select(F.col("alias_code").alias("_alias"),
                                                        F.col("product_code").alias("_canonical"))
VALID_PRODUCTS = spark.table("silver_products").select(F.col("product_code").alias("_valid_code"))


def norm_product(df, c="product_code"):
    """Upper/trim the code, map legacy aliases to the canonical code and drop unknown products."""
    df = df.withColumn("_pc", F.upper(F.trim(F.col(c))))
    df = df.join(F.broadcast(ALIAS_DF), F.col("_pc") == F.col("_alias"), "left")
    df = df.withColumn(c, F.coalesce(F.col("_canonical"), F.col("_pc"))).drop("_pc", "_alias", "_canonical")
    return df.join(F.broadcast(VALID_PRODUCTS), F.col(c) == F.col("_valid_code"), "left_semi")


plants = (read_bronze("erp_plants")
          .withColumn("plant_code", upper_trim("plant_code"))
          .withColumn("monthly_capacity_units", F.col("monthly_capacity_units").cast("int"))
          .withColumn("opened_year", F.col("opened_year").cast("int"))
          .dropDuplicates(["plant_code"]))
write_table(plants, "silver_plants")

suppliers = (read_bronze("erp_suppliers")
             .withColumn("supplier_id", upper_trim("supplier_id"))
             .withColumn("quality_rating", F.col("quality_rating").cast("double"))
             .withColumn("esg_score", F.col("esg_score").cast("int"))
             .withColumn("status", lower_trim("status"))
             .withColumn("onboarded_date", F.to_date("onboarded_date", "yyyy-MM-dd"))
             .dropDuplicates(["supplier_id"]))
write_table(suppliers, "silver_suppliers")

components = (read_bronze("plm_components")
              .withColumn("component_code", upper_trim("component_code"))
              .withColumn("primary_supplier_id", upper_trim("primary_supplier_id"))
              .withColumn("standard_cost_usd", F.col("standard_cost_usd").cast("double"))
              .withColumn("recyclable_pct", F.col("recyclable_pct").cast("double"))
              .withColumn("weight_g", F.col("weight_g").cast("int"))
              .withColumn("approved_supplier_count", F.col("approved_supplier_count").cast("int"))
              .withColumn("is_critical", F.coalesce(to_bool("is_critical"), F.lit(False)))
              .dropDuplicates(["component_code"]))
write_table(components, "silver_components")

bom = (norm_product(read_bronze("plm_bom"))
       .withColumn("component_code", upper_trim("component_code"))
       .withColumn("quantity", F.col("quantity").cast("double"))
       .withColumn("bom_version", F.col("bom_version").cast("int"))
       .withColumn("effective_from", F.to_date("effective_from", "yyyy-MM-dd"))
       .withColumn("effective_to", F.to_date("effective_to", "yyyy-MM-dd"))
       .dropDuplicates(["product_code", "component_code", "bom_version"]))
write_table(bom, "silver_bom")

## CRM customer segments (pet-owner profile)

In [ ]:
customers = (read_bronze("crm_customer_segments")
             .withColumn("customer_id", upper_trim("customer_id"))
             .withColumn("region_code", norm_region("region_code"))
             .withColumn("pet_owner", to_bool("pet_owner"))
             .withColumn("household_size", F.col("household_size").cast("int"))
             .withColumn("signup_date", F.to_date("signup_date", "yyyy-MM-dd"))
             .withColumn("segment", lower_trim("segment"))
             .dropDuplicates(["customer_id"]))
write_table(customers, "silver_customers")

CRM_PET = spark.table("silver_customers").select(F.col("customer_id").alias("_crm_id"),
                                                 F.col("pet_owner").alias("_crm_pet"))


def add_pet_owner(df, text_col):
    """pet_owner = profile flag -> pet mention in text -> CRM segment -> false (+ pet_owner_source)."""
    df = df.join(F.broadcast(CRM_PET), F.col("customer_id") == F.col("_crm_id"), "left")
    df = (df.withColumn("_explicit", to_bool("pet_owner"))
            .withColumn("_text_pet", F.col(text_col).rlike(PET_PATTERN)))
    return (df.withColumn("pet_owner_source",
                          F.when(F.col("_explicit").isNotNull(), "profile_flag")
                           .when(F.col("_text_pet"), "text")
                           .when(F.col("_crm_pet").isNotNull(), "crm")
                           .otherwise("default"))
              .withColumn("pet_owner", F.coalesce(F.col("_explicit"),
                                                  F.when(F.col("_text_pet"), F.lit(True)),
                                                  F.col("_crm_pet"), F.lit(False)))
              .drop("_explicit", "_text_pet", "_crm_id", "_crm_pet"))

## Sales orders (daily product × region × channel)

In [ ]:
fx = spark.table("silver_regions").select(F.col("region_code").alias("_fx_region"), "fx_to_usd")
sales = norm_product(read_bronze("erp_sales_orders").dropDuplicates(["order_line_id"]))
sales = (sales
         .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
         .withColumn("region_code", norm_region("region_code"))
         .withColumn("channel", F.coalesce(lower_trim("channel"), F.lit("unknown")))
         .withColumn("plant_code", upper_trim("plant_code"))
         .withColumn("units", F.col("units").cast("int"))
         .withColumn("unit_price_local", F.col("unit_price_local").cast("double"))
         .withColumn("gross_revenue_local", F.col("gross_revenue_local").cast("double"))
         .withColumn("discount_pct", F.coalesce(F.col("discount_pct").cast("double"), F.lit(0.0)))
         .withColumn("net_revenue_usd", F.col("net_revenue_usd").cast("double"))
         .withColumn("currency", upper_trim("currency"))
         .join(F.broadcast(fx), F.col("region_code") == F.col("_fx_region"), "left")
         .withColumn("gross_revenue_usd", F.round(F.col("gross_revenue_local") * F.col("fx_to_usd"), 2))
         .withColumn("net_revenue_usd", F.coalesce(
             F.col("net_revenue_usd"),
             F.round(F.col("gross_revenue_usd") * (F.lit(1.0) - F.col("discount_pct") / 100.0), 2)))
         .drop("_fx_region", "fx_to_usd")
         .filter(F.col("order_date").isNotNull() & (F.col("units") > 0)))
write_table(sales, "silver_sales")

## E-commerce reviews – language, pet owner, topic, sentiment

In [ ]:
reviews = norm_product(read_bronze("ecommerce_reviews").dropDuplicates(["review_id"]))
reviews = (reviews
           .withColumn("review_date", F.to_date("review_date", "yyyy-MM-dd"))
           .withColumn("region_code", norm_region("region_code"))
           .withColumn("customer_id", upper_trim("customer_id"))
           .withColumn("platform", F.trim(F.col("platform")))
           .withColumn("rating", F.col("rating").cast("int"))
           .withColumn("helpful_votes", F.coalesce(F.col("helpful_votes").cast("int"), F.lit(0)))
           .withColumn("verified_purchase", F.coalesce(to_bool("verified_purchase"), F.lit(False)))
           .withColumn("_text", F.concat_ws(" ", F.col("title"), F.col("review_text")))
           .withColumn("_low", F.lower(F.col("_text"))))
reviews = add_language(reviews, "_text")
reviews = add_pet_owner(reviews, "_low")
reviews = reviews.withColumn("topic", topic_expr(F.col("_low")))
reviews = add_sentiment(reviews, F.coalesce((F.col("rating") - 3.0) / 2.0, F.lit(0.0)), F.col("_low"))
reviews = reviews.filter(F.col("review_date").isNotNull()).drop("_text", "_low")
write_table(reviews, "silver_reviews")

## Customer-service tickets – same enrichment (tickets have no star rating)

In [ ]:
tickets = norm_product(read_bronze("cs_tickets").dropDuplicates(["ticket_id"]))
tickets = (tickets
           .withColumn("created_at", F.to_timestamp("created_at", "yyyy-MM-dd'T'HH:mm:ss"))
           .withColumn("closed_at", F.to_timestamp("closed_at", "yyyy-MM-dd'T'HH:mm:ss"))
           .withColumn("region_code", norm_region("region_code"))
           .withColumn("customer_id", upper_trim("customer_id"))
           .withColumn("contact_channel", lower_trim("contact_channel"))
           .withColumn("issue_category", F.coalesce(lower_trim("issue_category"), F.lit("unclassified")))
           .withColumn("priority", lower_trim("priority"))
           .withColumn("status", lower_trim("status"))
           .withColumn("csat_score", F.col("csat_score").cast("int"))
           .withColumn("_text", F.concat_ws(" ", F.col("subject"), F.col("description")))
           .withColumn("_low", F.lower(F.col("_text"))))
tickets = add_language(tickets, "_text")
tickets = add_pet_owner(tickets, "_low")
tickets = tickets.withColumn("topic", topic_expr(F.col("_low")))
tickets = add_sentiment(tickets, F.lit(TICKET_RATING_COMPONENT), F.col("_low"))
tickets = tickets.filter(F.col("created_at").isNotNull()).drop("_text", "_low")
write_table(tickets, "silver_tickets")

## Warranty claims

In [ ]:
warranty = norm_product(read_bronze("warranty_claims").dropDuplicates(["claim_id"]))
warranty = (warranty
            .withColumn("claim_date", F.to_date("claim_date", "yyyy-MM-dd"))
            .withColumn("purchase_date", F.to_date("purchase_date", "yyyy-MM-dd"))
            .withColumn("region_code", norm_region("region_code"))
            .withColumn("customer_id", upper_trim("customer_id"))
            .withColumn("failed_component_code", upper_trim("failed_component_code"))
            .withColumn("failure_mode", lower_trim("failure_mode"))
            .withColumn("repair_action", lower_trim("repair_action"))
            .withColumn("claim_status", lower_trim("claim_status"))
            .withColumn("claim_cost_usd", F.col("claim_cost_usd").cast("double"))
            .withColumn("days_in_service", F.col("days_in_service").cast("int"))
            .withColumn("purchase_date", F.coalesce(F.col("purchase_date"), F.expr("date_sub(claim_date, days_in_service)")))
            .withColumn("days_in_service", F.coalesce(F.col("days_in_service"), F.datediff("claim_date", "purchase_date")))
            .join(F.broadcast(CRM_PET), F.col("customer_id") == F.col("_crm_id"), "left")
            .withColumn("pet_owner", F.coalesce(F.col("_crm_pet"), F.lit(False)))
            .drop("_crm_id", "_crm_pet")
            .filter(F.col("claim_date").isNotNull()))
write_table(warranty, "silver_warranty")

## Supply chain: supplier capacity (monthly) and component costs (monthly)

In [ ]:
capacity = (read_bronze("scm_supplier_capacity")
            .withColumn("month", F.to_date(F.concat(F.substring(F.trim(F.col("month")), 1, 7), F.lit("-01")), "yyyy-MM-dd"))
            .withColumn("supplier_id", upper_trim("supplier_id"))
            .withColumn("component_code", upper_trim("component_code"))
            .withColumn("capacity_units", F.col("capacity_units").cast("int"))
            .withColumn("committed_units", F.col("committed_units").cast("int"))
            .withColumn("lead_time_days", F.col("lead_time_days").cast("int"))
            .withColumn("alt_supplier_qualification_days", F.col("alt_supplier_qualification_days").cast("int"))
            .withColumn("alt_supplier_candidate_id", upper_trim("alt_supplier_candidate_id"))
            .withColumn("utilisation_pct", F.coalesce(
                F.col("utilisation_pct").cast("double"),
                F.round(F.lit(100.0) * F.col("committed_units") / F.col("capacity_units"), 1)))
            .dropDuplicates(["month", "supplier_id", "component_code"]))
write_table(capacity, "silver_supplier_capacity")

costs = (read_bronze("scm_component_costs")
         .withColumn("month", F.to_date("month", "yyyy-MM-dd"))
         .withColumn("component_code", upper_trim("component_code"))
         .withColumn("primary_supplier_id", upper_trim("primary_supplier_id"))
         .withColumn("unit_cost_usd", F.col("unit_cost_usd").cast("double"))
         .withColumn("cost_index", F.col("cost_index").cast("double"))
         .dropDuplicates(["month", "component_code"]))
write_table(costs, "silver_component_costs")

## Logistics shipments

In [ ]:
factor = F.lit(None).cast("double")
for mode, f in EMISSION_FACTORS_KG_PER_TKM.items():
    factor = F.when(F.col("transport_mode") == F.lit(mode), F.lit(f)).otherwise(factor)

shipments = norm_product(read_bronze("logistics_shipments").dropDuplicates(["shipment_id"]))
shipments = (shipments
             .withColumn("ship_date", F.to_date("ship_date", "yyyy-MM-dd"))
             .withColumn("delivery_date", F.to_date("delivery_date", "yyyy-MM-dd"))
             .withColumn("origin_plant_code", upper_trim("origin_plant_code"))
             .withColumn("destination_region_code", norm_region("destination_region_code"))
             .withColumn("transport_mode", lower_trim("transport_mode"))
             .withColumn("units", F.col("units").cast("int"))
             .withColumn("planned_transit_days", F.col("planned_transit_days").cast("int"))
             .withColumn("weight_kg", F.col("weight_kg").cast("double"))
             .withColumn("distance_km", F.col("distance_km").cast("double"))
             .withColumn("freight_cost_usd", F.col("freight_cost_usd").cast("double"))
             .withColumn("co2e_kg", F.coalesce(
                 F.col("co2e_kg").cast("double"),
                 F.round(F.col("weight_kg") / 1000.0 * F.col("distance_km") * factor, 2)))
             .filter(F.col("ship_date").isNotNull()))
write_table(shipments, "silver_shipments")

## Market signals – keep the latest published revision

In [ ]:
w_latest = Window.partitionBy("month", "region_code", "category", "metric").orderBy(F.col("published_date").desc())
market = (read_bronze("market_signals")
          .withColumn("month", F.to_date("month", "yyyy-MM-dd"))
          .withColumn("region_code", norm_region("region_code"))
          .withColumn("category", lower_trim("category"))
          .withColumn("metric", lower_trim("metric"))
          .withColumn("value", F.col("value").cast("double"))
          .withColumn("published_date", F.to_date("published_date", "yyyy-MM-dd"))
          .withColumn("_rn", F.row_number().over(w_latest))
          .filter(F.col("_rn") == 1)
          .drop("_rn"))
write_table(market, "silver_market_signals")

## Row counts

In [ ]:
SILVER_TABLES = ["silver_regions", "silver_product_aliases", "silver_products", "silver_plants", "silver_suppliers",
                 "silver_components", "silver_bom", "silver_customers", "silver_sales", "silver_reviews", "silver_tickets",
                 "silver_warranty", "silver_supplier_capacity", "silver_component_costs", "silver_shipments",
                 "silver_market_signals"]
for t in SILVER_TABLES:
    print(f"{t:<32}{spark.table(t).count():>12,}")